Basic imports

In [1]:
import re
import json

Spark imports and init

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("dic").getOrCreate()
sc = spark.sparkContext

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4047. Attempting port 4048.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4048. Attempting port 4049.
25/05/13 16:23:22 WARN Utils: Service 'SparkUI' could not bind on port 4049. Attempting port 4050.
25/05/13 1

Load dataset and stopwords

In [3]:
dataset = sc.textFile("hdfs:///user/dic25_shared/amazon-reviews/full/reviews_devset.json")

In [4]:
with open("stopwords.txt") as f:
    stopwords = set(line.strip() for line in f)

Define mapper function

In [5]:
def category_token_doc_count_mapper(line):
  review = json.loads(line)
  text = review["reviewText"].lower()
  tokens = re.split(r"[ \t\d\(\)\[\]\{\}\.\!\?\,;:\+=\-_\"'`~#@&\*%€$§\\\/]+", text)
  filtered = set([t for t in tokens if len(t) > 1 and t not in stopwords])

  result = [((review["category"], t), 1) for t in filtered]
  result.append((("CATEGORY", review["category"]), 1))

  return result

In [6]:
counts = dataset.flatMap(lambda x: category_token_doc_count_mapper(x)).reduceByKey(lambda x, y: x + y).sortByKey()

Extract the valuable information into separate RDDs

In [7]:
category_doc_counts = counts.filter(lambda x: (x[0][0] == "CATEGORY")).map(lambda x: (x[0][1], x[1])).reduceByKey(lambda x, y: x + y).sortByKey()
category_doc_counts.collect()

[('Apps_for_Android', 2638),
 ('Automotive', 1374),
 ('Baby', 916),
 ('Beauty', 2023),
 ('Book', 22507),
 ('CDs_and_Vinyl', 3749),
 ('Cell_Phones_and_Accessorie', 3447),
 ('Clothing_Shoes_and_Jewelry', 5749),
 ('Digital_Music', 836),
 ('Electronic', 7825),
 ('Grocery_and_Gourmet_Food', 1297),
 ('Health_and_Personal_Care', 2982),
 ('Home_and_Kitche', 4254),
 ('Kindle_Store', 3205),
 ('Movies_and_TV', 4607),
 ('Musical_Instrument', 500),
 ('Office_Product', 1243),
 ('Patio_Lawn_and_Garde', 994),
 ('Pet_Supplie', 1235),
 ('Sports_and_Outdoor', 3269),
 ('Tools_and_Home_Improvement', 1926),
 ('Toys_and_Game', 2253)]

In [8]:
category_token_doc_counts = counts.filter(lambda x: (x[0][0] != "CATEGORY")).reduceByKey(lambda x, y: x + y).sortByKey()
category_token_doc_counts.take(5)

[(('Apps_for_Android', 'aacute'), 2),
 (('Apps_for_Android', 'abc'), 1),
 (('Apps_for_Android', 'abcs'), 2),
 (('Apps_for_Android', 'abhors'), 1),
 (('Apps_for_Android', 'abilities'), 1)]

In [9]:
token_doc_counts = category_token_doc_counts.map(lambda x: (x[0][1], x[1])).reduceByKey(lambda x, y: x + y).sortByKey()
token_doc_counts.take(5)

[('>>', 3), ('>>>', 1), ('>>>>>>>', 1), ('>>allison', 1), ('>>cedric', 1)]

In [10]:
total_doc_count = category_doc_counts.map(lambda x: x[1]).reduce(lambda x, y: x + y)
total_doc_count

78829

Broadcast dictionaries used for each calculation

In [11]:
token_doc_counts_dict = token_doc_counts.collectAsMap()
category_doc_counts_dict = category_doc_counts.collectAsMap()
token_doc_counts_bc = sc.broadcast(token_doc_counts_dict)
category_doc_counts_bc = sc.broadcast(category_doc_counts_dict)

Define Chi-Square calculation mapper function

In [12]:
def chi_square_mapper(record):
    category = record[0][0]
    token = record[0][1]
    observed_count = record[1]

    token_doc_count = token_doc_counts_bc.value.get(token, 0)
    category_doc_count = category_doc_counts_bc.value.get(category, 0)

    A = observed_count
    B = token_doc_count - A
    C = category_doc_count - A
    D = total_doc_count - A - B - C

    numerator = (A * D - B * C) ** 2
    denominator = (A + B) * (A + C) * (B + D) * (C + D)

    if denominator == 0:
      chi_square = 0
    else:
      chi_square = total_doc_count * numerator / denominator

    return (category, (token, chi_square))

Run calculation

In [13]:
chi_square_values = category_token_doc_counts.map(lambda x: chi_square_mapper(x))
chi_square_values.take(5)

[('Apps_for_Android', ('aacute', 0.0018593705920044118)),
 ('Apps_for_Android', ('abc', 0.0013963047293539942)),
 ('Apps_for_Android', ('abcs', 13.771796994110204)),
 ('Apps_for_Android', ('abhors', 5.7987240594342175)),
 ('Apps_for_Android', ('abilities', 2.0232237425766435))]

Sort by the calculated value

In [14]:
sorted_chi_square = chi_square_values.sortBy(lambda x: x[1][1], ascending=False)
sorted_chi_square.take(5)

[('CDs_and_Vinyl', ('music', 13083.770814228063)),
 ('Movies_and_TV', ('dvd', 8746.742991979037)),
 ('CDs_and_Vinyl', ('albums', 6705.824908271496)),
 ('CDs_and_Vinyl', ('tracks', 6516.952651181357)),
 ('Book', ('reading', 6184.609280406393))]

Produce results

In [15]:
top_tokens_per_category = sorted_chi_square.groupByKey().mapValues(lambda x: sorted(x, key=lambda y: y[1], reverse=True)[:75]).sortByKey()
formatted_top_tokens = top_tokens_per_category.map(lambda x: f"<{x[0]}> {' '.join([f'{t[0]}:{t[1]:.4f}' for t in x[1]])}")
merged_top_terms = top_tokens_per_category.map(lambda x: x[1]).flatMap(lambda x: [(y[0], 1) for y in x]).distinct().sortByKey()
merged_dict_string = " ".join(merged_top_terms.map(lambda x: x[0]).collect())

with open("output_rdd.txt", "w") as f:
    for element in formatted_top_tokens.collect():
        f.write(str(element) + "\n")
    f.write(merged_dict_string)